In [ ]:
$ python main.py
Enter agent name (agent1/agent2/agent3): agent1
🔧 Creating agent: agent1 with MCP servers: ['math-mcp']
✅ Loaded tools from math-mcp
Enter your query: What is 2 + 3?
🤖 Agent Response: The result is 5.


Excellent — that’s the **right design choice** ✅.
You want the `DynamicAgentFactory` in a **separate reusable module** — so you can import it in different projects (LangGraph, LangChain, FastMCP orchestrators, etc.).

Here’s the clean modular structure 👇

---

## 📁 Folder Structure

```
project_root/
│
├── agent_factory/
│   ├── __init__.py
│   ├── dynamic_agent_factory.py   ← 🧠 The reusable factory
│
├── mcp_clients/
│   ├── __init__.py
│   ├── math-mcp_client.py
│   ├── weather-mcp_client.py
│   ├── pollution-mcp_client.py
│
├── configs/
│   ├── agent_config.py
│
├── main.py
```

---

## 🧠 `agent_factory/dynamic_agent_factory.py`

```python
# agent_factory/dynamic_agent_factory.py
import importlib
from typing import Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, Tool


class DynamicAgentFactory:
    """
    Dynamically creates and manages LangChain/LangGraph agents
    based on a provided configuration.

    Each agent can be linked to different MCP servers, LLM models,
    and system prompts dynamically.
    """

    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.registry = {}

    async def create_agent(self, name: str):
        """
        Create and register an agent dynamically based on config.
        """
        if name not in self.config:
            raise ValueError(f"Agent '{name}' not found in configuration.")

        agent_cfg = self.config[name]
        print(f"🔧 Creating agent: {name}")
        print(f"   ↳ LLM: {agent_cfg['llm_model']}")
        print(f"   ↳ MCP Servers: {agent_cfg['mcp_servers']}")

        # 1️⃣ Load LLM
        llm = ChatOpenAI(model=agent_cfg["llm_model"], temperature=0.5)

        # 2️⃣ Dynamically import tools from MCP clients
        tools = []
        for mcp_name in agent_cfg["mcp_servers"]:
            try:
                module = importlib.import_module(f"mcp_clients.{mcp_name}_client")
                if hasattr(module, "get_tools"):
                    mcp_tools = module.get_tools()
                    tools.extend(mcp_tools)
                    print(f"✅ Loaded {len(mcp_tools)} tools from {mcp_name}")
                else:
                    print(f"⚠️ No get_tools() function found in {mcp_name}_client")
            except ModuleNotFoundError:
                print(f"❌ MCP client not found: {mcp_name}_client")

        # 3️⃣ Initialize agent
        agent = initialize_agent(
            tools=tools,
            llm=llm,
            agent="zero-shot-react-description",
            verbose=True,
        )

        # 4️⃣ Store in registry
        self.registry[name] = {
            "llm": llm,
            "tools": tools,
            "agent": agent,
            "system_prompt": agent_cfg.get("system_prompt", ""),
        }

        return agent

    async def get_agent(self, name: str):
        """
        Retrieve an agent from the registry or create it if missing.
        """
        if name not in self.registry:
            await self.create_agent(name)
        return self.registry[name]["agent"]
```

---

## ⚙️ `configs/agent_config.py`

```python
# configs/agent_config.py
AGENT_CONFIG = {
    "agent1": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": "You are a math expert. Use math tools wisely.",
        "mcp_servers": ["math-mcp"]
    },
    "agent2": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": "You are a weather expert. Answer using real-time data.",
        "mcp_servers": ["weather-mcp"]
    },
    "agent3": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": "You are an environment analyst. Use pollution data tools.",
        "mcp_servers": ["pollution-mcp"]
    }
}
```

---

## 🧩 `mcp_clients/math-mcp_client.py`

```python
# mcp_clients/math-mcp_client.py
from langchain.agents import Tool

def add(a: int, b: int) -> int:
    return a + b

def multiply(a: int, b: int) -> int:
    return a * b

def get_tools():
    return [
        Tool(name="add", func=add, description="Add two integers"),
        Tool(name="multiply", func=multiply, description="Multiply two integers")
    ]
```

(You can follow the same for `weather-mcp_client.py` and `pollution-mcp_client.py`.)

---

## 🚀 `main.py`

```python
# main.py
import asyncio
from agent_factory.dynamic_agent_factory import DynamicAgentFactory
from configs.agent_config import AGENT_CONFIG

async def main():
    factory = DynamicAgentFactory(AGENT_CONFIG)

    print("\nAvailable agents:", ", ".join(AGENT_CONFIG.keys()))
    agent_name = input("Enter agent name to activate: ").strip()

    try:
        agent = await factory.get_agent(agent_name)
        query = input("Enter your query: ").strip()
        print("\n🤖 Running agent...\n")
        result = agent.run(query)
        print(f"🧠 Agent Response:\n{result}\n")
    except ValueError as e:
        print(f"❌ {e}")

if __name__ == "__main__":
    asyncio.run(main())
```

---

## ✅ How it Works

1. `DynamicAgentFactory` loads LLM + MCP tools dynamically.
2. It keeps a **registry** of active agents.
3. You can add new agents just by editing the config file.
4. Each agent can have its own:

   * LLM backend (e.g., `gpt-4o`, `claude`, etc.)
   * MCP connections
   * System prompt

---

Would you like me to extend this with **LangGraph support** (e.g., to connect these agents in a workflow or orchestrator graph)?
That would let, for example, `Agent1 → Agent2 → Agent3` pass context between them automatically.
